In [22]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv('Bengaluru_House_Data.csv')
# data.head()
# data.info()

# value counts on each column to check columns individually 
"""
for column in data.columns:
    print(data[column].value_counts())
    print('*'*20)
"""
# Dropping because they are not contributing to ML 
data.drop(columns=['area_type','availability','society','balcony'],inplace=True)

data['location'].value_counts()   # it shows the frequency of each values in the column 
data['location'] = data['location'].fillna('Sarjapur  Road')

data['size'] = data['size'].fillna('2 BHK')

data['bath'] = data['bath'].fillna(data['bath'].median())

data['Bhk'] = data['size'].str.split().str.get(0).astype(int)
data[data.Bhk > 20]   # outliers

def convertRange(x):

    temp = x.split('-')
    if len(temp) == 2:
        return (float(temp[0]) + float(temp[1]))/2
    try:
        return float(x)
    except:
        return None
    
data['total_sqft'] = data['total_sqft'].apply(convertRange)

# Price per square feet 
data['price_per_sqft'] = data['price']*100000/data['total_sqft']

data['location'].value_counts()

data['location'] = data['location'].apply(lambda x: x .strip())
location_count = data['location'].value_counts()

location_count_less_10 = location_count[location_count<=10]

data['location'] = data['location'].apply(lambda x : 'other' if x in location_count_less_10 else x)

# Outlier Detection and removal 
(data['total_sqft']/data['Bhk']).describe()
data = data[((data['total_sqft']/data['Bhk']) >= 300)]
 
def remove_outliers_sqft(df):
    df_output = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)

        st = np.std(subdf.price_per_sqft)

        gen_df = subdf[(subdf.price_per_sqft > (m-st)) & (subdf.price_per_sqft <= (m+st))]
        df_output = pd.concat([df_output, gen_df], ignore_index= True)
    return df_output
data = remove_outliers_sqft(data)

def bhk_outlier_remover(Df):
    exclude_indices = np.array([])
    for location, location_df in data.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in location_df.groupby('bhk'):
            bhk_stats[bhk] = {
                'mean' : np.mean(bhk_df.price_per_sqft),
                'std' : np.std(bhk_df.price_per_sqft),
                'count' : bhk_df. shape[0]
            }

        for bhk, bhk_df in location_df.groupby('bhk'):
            stats = bhk_stats.get(bhk-1)
            if stats and stats['count'] > 5:
                exclude_indices = np.append(exclude_indices, bhk_df[bhk_df.price_per_sqft < (stats['mean'])].index.values)
    return data.drop(exclude_indices, axis='index')

data.drop(columns=['size', 'price_per_sqft'], inplace=True)

data.to_csv('Cleaned_data.csv')

x = data.drop(columns=['price'])
y = data['price']

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline

x_train, x_test, y_Train, y_test = train_test_split(x,y, test_size=0.2, random_state=0)

# Column transformer for categorical features
column_trans = make_column_transformer(
    (OneHotEncoder(sparse_output=False), ['location']),
    remainder='passthrough'
)

# Model without deprecated parameter
lr = LinearRegression()

# Pipeline
pipe = make_pipeline(column_trans, StandardScaler(), lr)

# Train
pipe.fit(x_train, y_Train)

# Predict
y_pred_lr = pipe.predict(x_test)

r2 = r2_score(y_test, y_pred_lr)
print("R² Score:", r2)

# Applying Lasso 
lasso = Lasso()
pipe = make_pipeline(column_trans, StandardScaler(), lasso)
pipe.fit(x_train, y_Train)
y_pred_lasso = pipe.predict(x_test)
r2_score(y_test, y_pred_lasso)

# Applying Ridge 
ridge = Ridge()
pipe = make_pipeline(column_trans, StandardScaler(), ridge)
pipe.fit(x_train, y_Train)
y_pred_ridge = pipe.predict(x_test)
r2_score(y_test, y_pred_ridge)

print("Linear Regression R²:", r2_score(y_test, y_pred_lr))
print("Lasso Regression R²:", r2_score(y_test, y_pred_lasso))
print("Ridge Regression R²:", r2_score(y_test, y_pred_ridge))

import pickle
pickle.dump(pipe, open('RidgeModel.pkl','wb'))

R² Score: 0.8298767380225317
Linear Regression R²: 0.8298767380225317
Lasso Regression R²: 0.8225976333132023
Ridge Regression R²: 0.8298734741832591
